In [ ]:
from __future__ import annotations

import os
import warnings
from typing import Dict, Iterable, List, Mapping, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    auc,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

import optuna

warnings.filterwarnings("ignore")

data_fs_static_external = pd.read_csv("YOUR_PATH")
data_fs_static_train = pd.read_csv("YOUR_PATH")

scale = 'yes'
feature_space = ['feature_1', 'feature_2', ...]

X_external, y_external  = do_train_test_split(data_fs_static_external,feature_space,scale)
X_train, y_train  = do_train_test_split(data_fs_static_train,feature_space,scale)


In [ ]:
RANDOM_SEED = 42
N_SPLITS_INNER = 5        # CV folds for Optuna objective
N_SPLITS_THRESHOLD = 5    # CV folds for OOF threshold selection
N_TRIALS = 200
N_BOOTSTRAPS = 2000

FIG_DIR = "figures"
SHAP_DIR = "shap_values"
OUT_DIR = "outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Matplotlib defaults
# -----------------------------
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)

print("Ready.")

def as_numpy(y: Iterable) -> np.ndarray:
    y_arr = np.asarray(y)
    return y_arr.reshape(-1)

def safe_confusion(yt: np.ndarray, yp: np.ndarray) -> Tuple[int, int, int, int]:
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)

def ci_mean(vals: Iterable[float], alpha: float = 0.05) -> Tuple[float, float, float]:
    v = np.asarray(list(vals), dtype=float)
    lo = np.percentile(v, 100 * (alpha / 2))
    hi = np.percentile(v, 100 * (1 - alpha / 2))
    return float(v.mean()), float(lo), float(hi)

def stable_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))

def stable_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(rec, prec))

def summarize_bootstrap(boot: Mapping[str, List[float]]) -> pd.DataFrame:
    rows = []
    for k in ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc"]:
        vals = [v for v in boot[k] if not np.isnan(v)]
        m, lo, hi = ci_mean(vals)
        rows.append([k, m, lo, hi])
    return pd.DataFrame(rows, columns=["metric", "mean", "ci_low", "ci_high"])

def align_features(
    X_target: pd.DataFrame,
    X_reference: pd.DataFrame,
    fill_strategy: str = "mean",  # "mean" or "zero"
) -> pd.DataFrame:
    X_aligned = X_target.copy()
    missing = [c for c in X_reference.columns if c not in X_aligned.columns]
    if missing:
        if fill_strategy == "mean":
            fill_vals = X_reference[missing].mean()
            for c in missing:
                X_aligned[c] = float(fill_vals[c])
        elif fill_strategy == "zero":
            for c in missing:
                X_aligned[c] = 0.0
        else:
            raise ValueError(f"Unknown fill_strategy='{fill_strategy}'")
    return X_aligned.reindex(columns=X_reference.columns)


def align_subject_reference(
    source_df: pd.DataFrame,
    X: pd.DataFrame,
    subject_col: str = "subject_reference",
) -> np.ndarray:
    """
    Align patient identifiers to the exact rows in X.

    This deliberately fails if preprocessing appears to have reset the index
    after dropping/reordering rows, because positional patient assignment would
    then be unsafe.
    """
    if subject_col not in source_df.columns:
        raise KeyError(f"Required grouping column '{subject_col}' not found.")
    if source_df[subject_col].isna().any():
        raise ValueError(f"Grouping column '{subject_col}' contains missing values.")

    if X.index.equals(source_df.index):
        groups = source_df.loc[X.index, subject_col]
    elif (
        source_df.index.is_unique
        and X.index.is_unique
        and X.index.isin(source_df.index).all()
        and not X.index.equals(pd.RangeIndex(start=0, stop=len(X), step=1))
    ):
        groups = source_df.loc[X.index, subject_col]
    else:
        raise RuntimeError(
            "Could not safely align subject_reference to the processed model rows. "
            "Ensure do_train_test_split() preserves the original dataframe index "
            "when rows are filtered or reordered."
        )

    groups_np = groups.to_numpy()
    if len(groups_np) != len(X):
        raise RuntimeError("Patient-ID alignment produced an unexpected row count.")
    return groups_np


def patient_admission_tables(groups: Iterable, cohort: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return cohort-level and admissions-per-patient summaries."""
    g = pd.Series(as_numpy(groups), name="subject_reference")
    counts = g.value_counts(dropna=False)

    summary = pd.DataFrame(
        [{
            "cohort": cohort,
            "hospital_stays_n": int(len(g)),
            "unique_patients_n": int(counts.size),
            "patients_with_recurrent_admissions_n": int((counts > 1).sum()),
            "patients_with_recurrent_admissions_pct": float(100 * (counts > 1).mean()),
            "admissions_per_patient_mean": float(counts.mean()),
            "admissions_per_patient_sd": float(counts.std(ddof=1)) if len(counts) > 1 else 0.0,
            "admissions_per_patient_median": float(counts.median()),
            "admissions_per_patient_min": int(counts.min()),
            "admissions_per_patient_max": int(counts.max()),
        }]
    )

    distribution = (
        counts.value_counts()
        .sort_index()
        .rename_axis("admissions_per_patient")
        .reset_index(name="unique_patients_n")
    )
    distribution.insert(0, "cohort", cohort)
    distribution["patients_pct"] = 100 * distribution["unique_patients_n"] / counts.size

    return summary, distribution


def _cluster_rows(groups: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray]]:
    """Create row-index lists for each unique patient cluster."""
    groups = as_numpy(groups)
    if pd.isna(groups).any():
        raise ValueError("Patient groups contain missing values.")
    unique_groups = pd.unique(groups)
    rows = [np.flatnonzero(groups == g) for g in unique_groups]
    return unique_groups, rows

def get_oof_probabilities(
    model: Pipeline,
    X: pd.DataFrame,
    y: Iterable,
    groups: Iterable,
    n_splits: int = N_SPLITS_THRESHOLD,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Generate leakage-free OOF probabilities with patient-grouped folds."""
    y_np = as_numpy(y)
    groups_np = as_numpy(groups)

    if len(groups_np) != len(X):
        raise ValueError("groups must have the same length as X.")

    oof = np.full(shape=(len(X),), fill_value=np.nan, dtype=float)

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(cv.split(X, y_np, groups=groups_np), start=1):
        train_groups = set(groups_np[tr])
        val_groups = set(groups_np[va])
        overlap = train_groups.intersection(val_groups)
        if overlap:
            raise RuntimeError(
                f"Patient overlap detected in threshold OOF fold {fold}: "
                f"{len(overlap)} overlapping patient(s)."
            )

        m = clone(model)
        m.fit(X.iloc[tr], y_np[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs; check data/CV.")
    return oof

def thresholds_from_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    if len(thresh) == 0:
        return {"F1-optimal": 0.5, "MCC-optimal": 0.5, "Youden": 0.5}

    f1_vals = 2 * (prec * rec) / (prec + rec + 1e-12)
    t_f1 = float(thresh[int(np.nanargmax(f1_vals[:-1]))])

    mcc_vals = [matthews_corrcoef(y_true, (y_prob >= t).astype(int)) for t in thresh]
    t_mcc = float(thresh[int(np.nanargmax(mcc_vals))])

    youden_vals = []
    for t in thresh:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = safe_confusion(y_true, yp)
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        youden_vals.append(sens + spec - 1)
    t_youden = float(thresh[int(np.nanargmax(youden_vals))])

    return {"F1-optimal": t_f1, "MCC-optimal": t_mcc, "Youden": t_youden}


def bootstrap_metrics_fixed_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: Iterable,
    threshold: float,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """
    Patient-level cluster bootstrap.

    Patients are sampled with replacement; whenever a patient is sampled,
    all hospital stays belonging to that patient are included together.
    """
    rng = np.random.RandomState(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups_np = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(groups_np)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    keys = ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc","tn","fp","fn","tp"]
    M: Dict[str, List[float]] = {k: [] for k in keys}

    unique_groups, cluster_rows = _cluster_rows(groups_np)
    n_clusters = len(unique_groups)

    for _ in range(n_boot):
        sampled_cluster_positions = rng.randint(0, n_clusters, size=n_clusters)
        idx = np.concatenate([cluster_rows[j] for j in sampled_cluster_positions])

        yt = y_true[idx]
        pr = y_prob[idx]
        yp = (pr >= threshold).astype(int)

        tn, fp, fn, tp = safe_confusion(yt, yp)
        spec = tn / (tn + fp + 1e-12)
        sens = tp / (tp + fn + 1e-12)
        npv  = tn / (tn + fn + 1e-12)

        M["accuracy"].append(accuracy_score(yt, yp))
        M["precision"].append(precision_score(yt, yp, zero_division=0))
        M["recall"].append(recall_score(yt, yp, zero_division=0))
        M["specificity"].append(spec)
        M["sensitivity"].append(sens)
        M["npv"].append(npv)
        M["f1"].append(f1_score(yt, yp, zero_division=0))
        M["mcc"].append(matthews_corrcoef(yt, yp))

        M["auc"].append(stable_roc_auc(yt, pr))
        M["auprc"].append(stable_auprc(yt, pr))

        M["tn"].append(tn); M["fp"].append(fp); M["fn"].append(fn); M["tp"].append(tp)

    return M

def _logit(p: np.ndarray) -> np.ndarray:
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def calibration_in_the_large(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.intercept_[0])

def calibration_slope(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.coef_[0][0])


def bootstrap_calibration(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: Iterable,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Patient-level cluster bootstrap for calibration metrics."""
    rng = np.random.RandomState(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups_np = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(groups_np)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    brier_vals, citl_vals, slope_vals = [], [], []
    unique_groups, cluster_rows = _cluster_rows(groups_np)
    n_clusters = len(unique_groups)

    for _ in range(n_boot):
        sampled_cluster_positions = rng.randint(0, n_clusters, size=n_clusters)
        idx = np.concatenate([cluster_rows[j] for j in sampled_cluster_positions])

        yt = y_true[idx]
        pr = y_prob[idx]

        # Logistic calibration metrics require both outcome classes.
        if len(np.unique(yt)) < 2:
            continue

        brier_vals.append(float(brier_score_loss(yt, pr)))
        citl_vals.append(calibration_in_the_large(yt, pr))
        slope_vals.append(calibration_slope(yt, pr))

    if not brier_vals:
        raise RuntimeError("No valid patient-cluster bootstrap samples for calibration.")

    return {"brier": brier_vals, "citl": citl_vals, "slope": slope_vals}

def plot_and_save_roc(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = stable_roc_auc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
    plt.plot([0,1],[0,1],"--", linewidth=1)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve (Logistic Regression)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_roc.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def plot_and_save_pr(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    auprc_val = stable_auprc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(rec, prec, label=f"AUPRC={auprc_val:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall curve (Logistic Regression)")
    plt.legend(loc="lower left")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_pr.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def plot_and_save_calibration(y_true: np.ndarray, y_prob: np.ndarray, name: str, n_bins: int = 10) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="uniform")

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(prob_pred, prob_true, marker="o", linewidth=1, label="Model")
    plt.plot([0,1],[0,1],"--", linewidth=1, label="Perfect")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Calibration curve (Logistic Regression)")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_calibration.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")


def make_objective_optuna_lr(
    X: pd.DataFrame,
    y: pd.Series,
    groups: Iterable,
    n_splits: int = N_SPLITS_INNER,
    seed: int = RANDOM_SEED,
):
    """
    Optuna objective: mean patient-grouped CV ROC AUC.
    Uses leakage-free scaling via a Pipeline inside each fold.
    """
    y_np = as_numpy(y)
    groups_np = as_numpy(groups)

    if len(groups_np) != len(X):
        raise ValueError("groups must have the same length as X.")

    def objective(trial: optuna.trial.Trial) -> float:
        penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])
        C = trial.suggest_float("C", 1e-4, 10.0, log=True)

        if penalty == "elasticnet":
            l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
        else:
            l1_ratio = None

        pipe = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "lr",
                    LogisticRegression(
                        penalty=penalty,
                        C=C,
                        l1_ratio=l1_ratio,
                        solver="saga",
                        class_weight="balanced",
                        max_iter=5000,
                        n_jobs=-1,
                        random_state=seed,
                    ),
                ),
            ]
        )

        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        aucs = []
        for fold, (tr, va) in enumerate(
            cv.split(X, y_np, groups=groups_np), start=1
        ):
            overlap = set(groups_np[tr]).intersection(set(groups_np[va]))
            if overlap:
                raise RuntimeError(
                    f"Patient overlap detected in Optuna CV fold {fold}: "
                    f"{len(overlap)} overlapping patient(s)."
                )

            pipe.fit(X.iloc[tr], y_np[tr])
            prob = pipe.predict_proba(X.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y_np[va], prob))
        return float(np.mean(aucs))

    return objective

def build_lr_pipeline_from_params(params: Dict, seed: int = RANDOM_SEED) -> Pipeline:
    penalty = params["penalty"]
    C = params["C"]
    l1_ratio = params.get("l1_ratio", None) if penalty == "elasticnet" else None

    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "lr",
                LogisticRegression(
                    penalty=penalty,
                    C=C,
                    l1_ratio=l1_ratio,
                    solver="saga",
                    class_weight="balanced",
                    max_iter=5000,
                    n_jobs=-1,
                    random_state=seed,
                ),
            ),
        ]
    )



def run_pipeline_lr_external(
    X_train: pd.DataFrame,
    y_train: Iterable,
    groups_train: Iterable,
    X_external: pd.DataFrame,
    y_external: Iterable,
    groups_external: Iterable,
    feature_fill_strategy: str = "mean",
    n_trials: int = N_TRIALS,
    n_splits_inner: int = N_SPLITS_INNER,
    n_splits_threshold: int = N_SPLITS_THRESHOLD,
    n_bootstraps: int = N_BOOTSTRAPS,
    run_shap: bool = True,
    prefix: str = "logreg_external",
) -> Dict[str, Dict[str, List[float]]]:
    """
    External validation with patient-level grouping where required.

    Steps:
      1) Patient-grouped Optuna CV tuning on development data
      2) Fit tuned model on all development data
      3) Choose thresholds from patient-grouped TRAINING OOF probabilities
      4) Predict the complete external cohort
      5) Estimate external CIs using patient-level cluster bootstrap
      6) Calibration analysis + patient-cluster bootstrap CIs
      7) SHAP (optional)
    """
    y_train_np = as_numpy(y_train)
    y_ext_np = as_numpy(y_external)
    groups_train_np = as_numpy(groups_train)
    groups_ext_np = as_numpy(groups_external)

    if len(X_train) != len(y_train_np) or len(X_train) != len(groups_train_np):
        raise ValueError("Training X, y, and patient groups are not aligned.")
    if len(X_external) != len(y_ext_np) or len(X_external) != len(groups_ext_np):
        raise ValueError("External X, y, and patient groups are not aligned.")

    if "subject_reference" in X_train.columns or "subject_reference" in X_external.columns:
        raise ValueError(
            "subject_reference must be used only as a grouping variable, not as a predictor."
        )

    # Save patient/admission summaries for transparent reporting.
    train_summary, train_dist = patient_admission_tables(groups_train_np, "development")
    ext_summary, ext_dist = patient_admission_tables(groups_ext_np, "external")
    cohort_summary = pd.concat([train_summary, ext_summary], ignore_index=True)
    admission_distribution = pd.concat([train_dist, ext_dist], ignore_index=True)

    cohort_summary.to_csv(
        os.path.join(OUT_DIR, f"{prefix}_patient_admission_summary.csv"), index=False
    )
    admission_distribution.to_csv(
        os.path.join(OUT_DIR, f"{prefix}_admissions_per_patient_distribution.csv"),
        index=False,
    )

    print("Patient/admission summary:")
    display(cohort_summary)

    # ---- 1) Patient-grouped Optuna tuning
    print("Running patient-grouped Optuna hyperparameter tuning (AUC)...")
    study = optuna.create_study(direction="maximize")
    study.optimize(
        make_objective_optuna_lr(
            X_train,
            pd.Series(y_train_np),
            groups=groups_train_np,
            n_splits=n_splits_inner,
        ),
        n_trials=n_trials,
        show_progress_bar=False,
    )
    best_params = study.best_params
    best_value = study.best_value

    print("\nBest parameters:")
    print(best_params)
    print(f"Best grouped CV AUC: {best_value:.4f}")

    best_path = os.path.join(OUT_DIR, f"{prefix}_best_params.json")
    pd.Series(best_params).to_json(best_path)
    print(f"Saved: {best_path}")

    # ---- 2) Fit tuned model on all development data
    model = build_lr_pipeline_from_params(best_params)
    print("\nFitting final tuned Logistic Regression on all training data...")
    model.fit(X_train, y_train_np)

    # ---- 3) Threshold selection from patient-grouped TRAINING OOF probabilities
    print("\nComputing patient-grouped training OOF probabilities for threshold selection...")
    model_for_oof = build_lr_pipeline_from_params(best_params)
    p_oof = get_oof_probabilities(
        model_for_oof,
        X_train,
        y_train_np,
        groups=groups_train_np,
        n_splits=n_splits_threshold,
    )
    thresholds = thresholds_from_predictions(y_train_np, p_oof)

    thr_df = pd.DataFrame(
        {"rule": list(thresholds.keys()), "threshold": list(thresholds.values())}
    )
    thr_path = os.path.join(OUT_DIR, f"{prefix}_thresholds_from_train_oof.csv")
    thr_df.to_csv(thr_path, index=False)

    print("\nFrozen thresholds (derived from patient-grouped TRAINING OOF predictions):")
    display(thr_df)
    print(f"Saved: {thr_path}")

    # ---- 4) External prediction; external cohort remains intact
    X_ext = align_features(X_external, X_train, fill_strategy=feature_fill_strategy)
    p_ext = model.predict_proba(X_ext)[:, 1]

    print("\nExternal ROC/PR/Calibration plots:")
    plot_and_save_roc(y_ext_np, p_ext, name=prefix)
    plot_and_save_pr(y_ext_np, p_ext, name=prefix)
    plot_and_save_calibration(y_ext_np, p_ext, name=prefix)

    # ---- 5) External performance with frozen thresholds + patient-cluster bootstrap
    results: Dict[str, Dict[str, List[float]]] = {}
    summaries = []

    for rule, thr in thresholds.items():
        boot = bootstrap_metrics_fixed_threshold(
            y_ext_np,
            p_ext,
            groups=groups_ext_np,
            threshold=thr,
            n_boot=n_bootstraps,
        )
        results[rule] = boot

        df = summarize_bootstrap(boot)
        df.insert(0, "rule", rule)
        df.insert(1, "threshold", thr)
        summaries.append(df)

    perf_df = pd.concat(summaries, ignore_index=True)
    perf_path = os.path.join(
        OUT_DIR, f"{prefix}_external_patient_cluster_bootstrap_metrics.csv"
    )
    perf_df.to_csv(perf_path, index=False)

    print("\nExternal performance summary (patient-cluster bootstrap mean + 95% CI):")
    display(perf_df)
    print(f"Saved: {perf_path}")

    # ---- 6) Calibration (external)
    print("\nCalibration analysis (External):")
    brier = float(brier_score_loss(y_ext_np, p_ext))
    citl = calibration_in_the_large(y_ext_np, p_ext)
    slope = calibration_slope(y_ext_np, p_ext)
    print(f"Brier score (point):       {brier:.4f}")
    print(f"CITL (point):              {citl:.4f}")
    print(f"Calibration slope (point): {slope:.4f}")

    C = bootstrap_calibration(
        y_ext_np,
        p_ext,
        groups=groups_ext_np,
        n_boot=n_bootstraps,
    )
    cal_rows = []
    for label, key in [("Brier", "brier"), ("CITL", "citl"), ("Slope", "slope")]:
        m, lo, hi = ci_mean(C[key])
        cal_rows.append([label, m, lo, hi])

    cal_df = pd.DataFrame(
        cal_rows, columns=["metric", "mean", "ci_low", "ci_high"]
    )
    cal_path = os.path.join(
        OUT_DIR, f"{prefix}_external_calibration_patient_cluster_bootstrap.csv"
    )
    cal_df.to_csv(cal_path, index=False)

    display(cal_df)
    print(f"Saved: {cal_path}")

    # ---- 7) SHAP (optional)
    if run_shap:
        print("\nRunning SHAP (LR tuned, scaled space)...")
        run_shap_for_logreg(model, X_train, X_ext, prefix="logreg")

    print("\nDone.")
    return results

def subsample_df(X: pd.DataFrame, n: int, seed: int = RANDOM_SEED) -> pd.DataFrame:
    if len(X) <= n:
        return X
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X), n, replace=False)
    return X.iloc[idx]

def run_shap_for_logreg(
    model: Pipeline,
    X_train: pd.DataFrame,
    X_external_aligned: pd.DataFrame,
    max_shap_samples: int = 500,
    background_n: int = 200,
    seed: int = RANDOM_SEED,
    prefix: str = "logreg",
) -> None:
    """
    SHAP LinearExplainer in scaled feature space of StandardScaler -> LogisticRegression.
    Saves mean(|SHAP|) per feature CSVs + summary plots.
    """
    try:
        import shap
    except ModuleNotFoundError:
        print("SHAP not installed. Install with: pip install shap")
        return

    scaler: StandardScaler = model.named_steps["scaler"]
    lr: LogisticRegression = model.named_steps["lr"]

    X_train_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=X_train.columns, index=X_train.index
    )
    X_ext_scaled = pd.DataFrame(
        scaler.transform(X_external_aligned), columns=X_train.columns, index=X_external_aligned.index
    )

    X_train_shap = subsample_df(X_train_scaled, max_shap_samples, seed)
    X_ext_shap = subsample_df(X_ext_scaled, max_shap_samples, seed)

    bg_n = min(background_n, len(X_train_scaled))
    background = shap.sample(X_train_scaled, bg_n, random_state=seed)

    explainer = shap.LinearExplainer(lr, background)
    shap_train = np.asarray(explainer.shap_values(X_train_shap))
    shap_ext = np.asarray(explainer.shap_values(X_ext_shap))

    def save_mean_abs(shap_matrix: np.ndarray, feature_names: List[str], name: str) -> str:
        mean_abs = np.abs(shap_matrix).mean(axis=0)
        df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs}).sort_values(
            "mean_abs_shap", ascending=False
        )
        out = os.path.join(SHAP_DIR, f"{prefix}_shap_{name}.csv")
        df.to_csv(out, index=False)
        return out

    p1 = save_mean_abs(shap_train, list(X_train.columns), "train_scaled")
    p2 = save_mean_abs(shap_ext, list(X_train.columns), "external_scaled")
    print(f"Saved: {p1}")
    print(f"Saved: {p2}")

    # Summary plots saved + inline
    plt.figure()
    shap.summary_plot(shap_train, X_train_shap, show=False)
    plt.tight_layout()
    out1 = os.path.join(FIG_DIR, f"{prefix}_shap_summary_train.png")
    plt.savefig(out1)
    plt.show()
    plt.close()
    print(f"Saved: {out1}")

    plt.figure()
    shap.summary_plot(shap_ext, X_ext_shap, show=False)
    plt.tight_layout()
    out2 = os.path.join(FIG_DIR, f"{prefix}_shap_summary_external.png")
    plt.savefig(out2)
    plt.show()
    plt.close()
    print(f"Saved: {out2}")

# Requires:
# data_fs_static_train / data_fs_static_external containing subject_reference,
# and X/y objects produced above. The model matrices must preserve source row indices.

groups_train = align_subject_reference(
    data_fs_static_train, X_train, subject_col="subject_reference"
)
groups_external = align_subject_reference(
    data_fs_static_external, X_external, subject_col="subject_reference"
)

results_logreg = run_pipeline_lr_external(
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    X_external=X_external,
    y_external=y_external,
    groups_external=groups_external,
    feature_fill_strategy="mean",
    n_trials=N_TRIALS,
    n_splits_inner=N_SPLITS_INNER,
    n_splits_threshold=N_SPLITS_THRESHOLD,
    n_bootstraps=N_BOOTSTRAPS,
    run_shap=True,
    prefix="logreg_external",
)
